<a href="https://colab.research.google.com/github/mbithi002/generativeai/blob/main/mbithi002_implementation_research_logs_on_Transformer_Architecture_and_attention_based_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TRANSFORMER ARCHITECTURE & ATTENTION VISUALIZATION

1. Install Requirements

In [ ]:
!pip install transformers torch matplotlib seaborn plotly bertviz

2. Import packages

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, GPT2LMHeadModel, GPT2Tokenizer
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("TRANSFORMER ARCHITECTURE & ATTENTION VISUALIZATION")
print("="*60)

# Part 1: Load Models

In [ ]:
print("\n📥 Loading models...")

# Load BERT (Encoder-only) for understanding tasks
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)

# Load GPT2 (Decoder-only) for generation tasks
gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2", output_attentions=True)
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token

print("✅ Models loaded successfully!")

# PART 2: Visualize token attention

In [ ]:
print("\n" + "="*60)
print("PART 2: VISUALIZING TOKEN ATTENTION")
print("="*60)

def visualize_attention(text, model, tokenizer, model_type="bert", layer=0, head=0):
    """Visualize attention patterns for a given text"""

    # Tokenize input
    inputs = tokenizer(text, return_tensors="pt")

    # Get model outputs with attention
    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)

    # Extract attention weights
    if model_type == "bert":
        # BERT: shape (layers, batch, heads, seq_len, seq_len)
        attentions = outputs.attentions
        tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    else:
        # GPT2: shape (layers, batch, heads, seq_len, seq_len)
        attentions = outputs.attentions
        tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    # Get specific layer and head
    attention_matrix = attentions[layer][0, head].numpy()

    # Create attention visualization
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    # Plot 1: Attention heatmap
    im = axes[0].imshow(attention_matrix, cmap='viridis', aspect='auto')
    axes[0].set_xticks(range(len(tokens)))
    axes[0].set_yticks(range(len(tokens)))
    axes[0].set_xticklabels(tokens, rotation=45, ha='right')
    axes[0].set_yticklabels(tokens)
    axes[0].set_title(f'Attention Pattern - Layer {layer}, Head {head}')
    plt.colorbar(im, ax=axes[0])

    # Plot 2: Attention for a specific token
    token_idx = len(tokens) // 2  # Middle token
    attention_row = attention_matrix[token_idx]

    bars = axes[1].bar(range(len(tokens)), attention_row)
    axes[1].set_xticks(range(len(tokens)))
    axes[1].set_xticklabels(tokens, rotation=45, ha='right')
    axes[1].set_title(f'Attention from "{tokens[token_idx]}" to all tokens')
    axes[1].set_xlabel('Tokens')
    axes[1].set_ylabel('Attention Weight')

    # Color the bar for the token itself differently
    bars[token_idx].set_color('red')

    plt.tight_layout()
    plt.show()

    return attention_matrix, tokens

# Test with different sentences
print("\n📊 Visualizing Attention for Different Sentences:\n")

# Example 1: Simple sentence with clear relationships
text1 = "The cat sat on the mat"
print(f"Sentence 1: '{text1}'")
attn1, tokens1 = visualize_attention(text1, bert_model, bert_tokenizer, "bert", layer=0, head=0)

# Example 2: Sentence with pronoun reference
text2 = "The animal didn't cross the street because it was tired"
print(f"\nSentence 2: '{text2}'")
attn2, tokens2 = visualize_attention(text2, bert_model, bert_tokenizer, "bert", layer=5, head=3)


# PART 3: Multi-head attention visualization

In [ ]:
print("\n" + "="*60)
print("PART 3: MULTI-HEAD ATTENTION VISUALIZATION")
print("="*60)

def visualize_multihead_attention(text, model, tokenizer, layer=0):
    """Visualize multiple attention heads"""

    inputs = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)

    attentions = outputs.attentions[layer][0].numpy()  # (heads, seq_len, seq_len)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    # Create subplot grid
    n_heads = attentions.shape[0]
    grid_size = int(np.ceil(np.sqrt(n_heads)))

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(15, 15))
    fig.suptitle(f'Multi-Head Attention - Layer {layer}', fontsize=16)

    for head_idx in range(n_heads):
        row = head_idx // grid_size
        col = head_idx % grid_size

        ax = axes[row, col]
        im = ax.imshow(attentions[head_idx], cmap='viridis', aspect='auto')
        ax.set_title(f'Head {head_idx}')

        if row == grid_size - 1:
            ax.set_xticks(range(len(tokens)))
            ax.set_xticklabels(tokens, rotation=90, fontsize=8)
        else:
            ax.set_xticks([])

        if col == 0:
            ax.set_yticks(range(len(tokens)))
            ax.set_yticklabels(tokens, fontsize=8)
        else:
            ax.set_yticks([])

    plt.tight_layout()
    plt.show()

print("\n📊 Visualizing Multiple Attention Heads:")
visualize_multihead_attention("The quick brown fox jumps", bert_model, bert_tokenizer, layer=0)


# PART 4: Encoder vs Decoder Attention Patterns

In [ ]:
print("\n" + "="*60)
print("PART 4: ENCODER vs DECODER ATTENTION PATTERNS")
print("="*60)

def compare_attention_patterns(text):
    """Compare BERT (bidirectional) vs GPT2 (causal) attention"""

    print(f"\nText: '{text}'\n")

    # BERT (Encoder - Bidirectional)
    bert_inputs = bert_tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        bert_outputs = bert_model(**bert_inputs, output_attentions=True)

    bert_attn = bert_outputs.attentions[0][0, 0].numpy()
    bert_tokens = bert_tokenizer.convert_ids_to_tokens(bert_inputs['input_ids'][0])

    # GPT2 (Decoder - Causal/Unidirectional)
    gpt2_inputs = gpt2_tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        gpt2_outputs = gpt2_model(**gpt2_inputs, output_attentions=True)

    gpt2_attn = gpt2_outputs.attentions[0][0, 0].numpy()
    gpt2_tokens = gpt2_tokenizer.convert_ids_to_tokens(gpt2_inputs['input_ids'][0])

    # Plot comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # BERT (Bidirectional)
    im1 = axes[0].imshow(bert_attn, cmap='viridis', aspect='auto')
    axes[0].set_xticks(range(len(bert_tokens)))
    axes[0].set_yticks(range(len(bert_tokens)))
    axes[0].set_xticklabels(bert_tokens, rotation=45, ha='right')
    axes[0].set_yticklabels(bert_tokens)
    axes[0].set_title('BERT (Encoder) - Bidirectional\nEach token sees ALL tokens')
    plt.colorbar(im1, ax=axes[0])

    # GPT2 (Causal)
    im2 = axes[1].imshow(gpt2_attn, cmap='plasma', aspect='auto')
    axes[1].set_xticks(range(len(gpt2_tokens)))
    axes[1].set_yticks(range(len(gpt2_tokens)))
    axes[1].set_xticklabels(gpt2_tokens, rotation=45, ha='right')
    axes[1].set_yticklabels(gpt2_tokens)
    axes[1].set_title('GPT2 (Decoder) - Causal\nEach token ONLY sees previous tokens')
    plt.colorbar(im2, ax=axes[1])

    # Add triangle to show causal masking
    for i in range(len(gpt2_tokens)):
        for j in range(len(gpt2_tokens)):
            if j > i:  # Future tokens
                axes[1].add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                                               fill=True, color='black', alpha=0.3))

    plt.tight_layout()
    plt.show()

    print("🔍 Observation:")
    print("  • BERT: Full matrix - each token attends to all tokens")
    print("  • GPT2: Lower triangular - each token only attends to previous tokens")

compare_attention_patterns("The cat sat on the mat")

#PART 5: Positional Encoding Visualization

In [ ]:
print("\n" + "="*60)
print("PART 5: POSITIONAL ENCODING VISUALIZATION")
print("="*60)

def visualize_positional_encoding(max_len=50, d_model=16):
    """Visualize sinusoidal positional encodings"""

    def get_positional_encoding(max_len, d_model):
        pe = np.zeros((max_len, d_model))
        position = np.arange(0, max_len).reshape(-1, 1)
        div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))

        pe[:, 0::2] = np.sin(position * div_term)
        pe[:, 1::2] = np.cos(position * div_term)
        return pe

    # Get positional encodings
    pe = get_positional_encoding(max_len, d_model)

    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    # Plot 1: Heatmap of full positional encoding
    im1 = axes[0, 0].imshow(pe.T, aspect='auto', cmap='RdBu')
    axes[0, 0].set_xlabel('Position')
    axes[0, 0].set_ylabel('Dimension')
    axes[0, 0].set_title('Positional Encoding Heatmap')
    plt.colorbar(im1, ax=axes[0, 0])

    # Plot 2: Values for specific dimensions
    for dim in [0, 3, 7, 15]:
        if dim < d_model:
            axes[0, 1].plot(pe[:, dim], label=f'Dim {dim}')
    axes[0, 1].set_xlabel('Position')
    axes[0, 1].set_ylabel('Encoding Value')
    axes[0, 1].set_title('Positional Encoding Values by Dimension')
    axes[0, 1].legend()
    axes[0, 1].grid(True)

    # Plot 3: 3D visualization of first 3 dimensions
    from mpl_toolkits.mplot3d import Axes3D
    ax3d = fig.add_subplot(2, 2, 3, projection='3d')

    positions = range(min(30, max_len))
    xs = pe[positions, 0]
    ys = pe[positions, 1]
    zs = pe[positions, 2]

    ax3d.scatter(xs, ys, zs, c=positions, cmap='viridis', s=50)
    ax3d.set_xlabel('Dim 0')
    ax3d.set_ylabel('Dim 1')
    ax3d.set_zlabel('Dim 2')
    ax3d.set_title('Position Encoding in 3D Space')

    # Plot 4: Similarity between positions
    similarity = np.dot(pe, pe.T)
    im4 = axes[1, 1].imshow(similarity, cmap='viridis', aspect='auto')
    axes[1, 1].set_xlabel('Position')
    axes[1, 1].set_ylabel('Position')
    axes[1, 1].set_title('Position Similarity Matrix')
    plt.colorbar(im4, ax=axes[1, 1])

    plt.tight_layout()
    plt.show()

    print("🔍 Key Observations:")
    print("  • Each position gets a unique encoding")
    print("  • Nearby positions have similar encodings")
    print("  • The pattern repeats at different frequencies")

visualize_positional_encoding()

# PART 6: Interactive Attention Visualization

In [ ]:
print("\n" + "="*60)
print("PART 6: INTERACTIVE ATTENTION VISUALIZER")
print("="*60)

def interactive_attention_visualizer():
    """Create an interactive visualization using plotly"""

    # Sample sentence
    text = "The animal didn't cross the street because it was tired"
    inputs = bert_tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        outputs = bert_model(**inputs, output_attentions=True)

    # Get attention from layer 5, head 3
    attention = outputs.attentions[5][0, 3].numpy()
    tokens = bert_tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    # Create interactive heatmap
    fig = go.Figure(data=go.Heatmap(
        z=attention,
        x=tokens,
        y=tokens,
        colorscale='Viridis',
        hoverongaps=False,
        text=[[f'From "{tokens[i]}" to "{tokens[j]}": {attention[i,j]:.3f}'
               for j in range(len(tokens))] for i in range(len(tokens))],
        hoverinfo='text'
    ))

    fig.update_layout(
        title='Interactive Attention Heatmap (Layer 5, Head 3)',
        xaxis_title='Target Token',
        yaxis_title='Source Token',
        width=700,
        height=700
    )

    fig.show()

    # Create bar chart for specific token
    token_idx = tokens.index('it') if 'it' in tokens else len(tokens)//2

    fig2 = go.Figure(data=go.Bar(
        x=tokens,
        y=attention[token_idx],
        marker_color=['red' if i == token_idx else 'blue' for i in range(len(tokens))]
    ))

    fig2.update_layout(
        title=f'Attention from "{tokens[token_idx]}" to all tokens',
        xaxis_title='Tokens',
        yaxis_title='Attention Weight',
        width=800,
        height=400
    )

    fig2.show()

interactive_attention_visualizer()